# Question 1: Employment Statistics Analysis

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Question:** Using the NYS DOL's employment statistics data for the latest available month:
- **1a)** Which major industries (by 2-digit NAICS) in New York City changed the most over the prior year? Describe possible reasons why.
- **1b)** How was the 1-year change different from the 5-year change?

**Data Source:** NYS DOL Current Employment Statistics (`ces.csv`), not seasonally adjusted.
**Latest Available Month:** February 2026.
**Geography:** New York City (5 boroughs).

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ces = pd.read_csv('../data/ces.csv')
ces.columns = ces.columns.str.strip()
nyc = ces[ces['AREANAME'] == 'New York City'].copy()

---
## Step 2.1–2.3: Prepare Data

Filter to 2-digit NAICS sectors for NYC. Extract Feb 2026, Feb 2025, and Feb 2021 employment.

**Data notes:**
- ces.csv is **not seasonally adjusted** — we only compare same-month across years
- Employment figures are in **actual units** (not thousands)
- NAICS 21 (Mining) and 23 (Construction) are **combined** in the data
- Education (61) and Health Care (62) are **private sector only**

In [ ]:
sectors = {
    '21+23': 'Mining, Logging and Construction',
    '31-33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44-45': 'Retail Trade',
    '48-49': 'Transportation and Warehousing',
    '22': 'Utilities',
    '51': 'Information',
    '52': 'Finance and Insurance',
    '53': 'Real Estate and Rental and Leasing',
    '54': 'Professional, Scientific, and Technical Services',
    '55': 'Management of Companies and Enterprises',
    '56': 'Administrative and Support and Waste Management and Remediation Services',
    '61': 'Private Educational Services',
    '62': 'Health Care and Social Assistance',
    '71': 'Arts, Entertainment, and Recreation',
    '72': 'Accommodation and Food Services',
    '81': 'Other Services',
    '92': 'Government',
}

short = {
    'Mining, Logging and Construction': 'Mining/Construction',
    'Manufacturing': 'Manufacturing',
    'Wholesale Trade': 'Wholesale Trade',
    'Retail Trade': 'Retail Trade',
    'Transportation and Warehousing': 'Transport/Warehousing',
    'Utilities': 'Utilities',
    'Information': 'Information',
    'Finance and Insurance': 'Finance/Insurance',
    'Real Estate and Rental and Leasing': 'Real Estate',
    'Professional, Scientific, and Technical Services': 'Prof/Technical Svcs',
    'Management of Companies and Enterprises': 'Management of Cos',
    'Administrative and Support and Waste Management and Remediation Services': 'Admin/Waste Mgmt',
    'Private Educational Services': 'Education (Private)',
    'Health Care and Social Assistance': 'Health Care/Social Asst',
    'Arts, Entertainment, and Recreation': 'Arts/Entertainment',
    'Accommodation and Food Services': 'Accommodation/Food',
    'Other Services': 'Other Services',
    'Government': 'Government',
}

rows = []
for naics, title in sectors.items():
    feb26 = nyc[(nyc['YEAR'] == 2026) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    feb25 = nyc[(nyc['YEAR'] == 2025) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    feb21 = nyc[(nyc['YEAR'] == 2021) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    if len(feb26) > 0 and len(feb25) > 0 and len(feb21) > 0:
        v26, v25, v21 = feb26.values[0], feb25.values[0], feb21.values[0]
        if pd.notna(v26) and pd.notna(v25) and pd.notna(v21):
            rows.append({
                'NAICS': naics, 'Industry': title,
                'Feb 2026': int(v26), 'Feb 2025': int(v25), 'Feb 2021': int(v21),
                'YoY Change': int(v26 - v25), 'YoY %': round((v26 - v25) / v25 * 100, 2),
                '5yr Change': int(v26 - v21), '5yr %': round((v26 - v21) / v21 * 100, 2),
            })

df = pd.DataFrame(rows)
print(f'Sectors with complete data: {len(df)}')

---
## Step 2.4–2.5: Year-over-Year Changes (Q1a)

Rank industries by magnitude of change. We present both absolute and percentage change.

### BLS Supersector Groupings

The grouped sectors below follow the **BLS NAICS supersector classification** defined at:
https://www.bls.gov/sae/additional-resources/naics-supersectors-for-ces-program.htm

BLS aggregates NAICS sectors into supersectors for analysis. The groupings used here are:
- **Leisure & Hospitality** (Supersector 70): NAICS 71 + 72
- **Education & Health Services** (Supersector 65): NAICS 61 + 62
- **Trade, Transportation & Utilities** (Supersector 40): NAICS 22 + 42 + 44-45 + 48-49
- **Professional & Business Services** (Supersector 60): NAICS 54 + 55 + 56
- **Financial Activities** (Supersector 55): NAICS 52 + 53

In [ ]:
super_groups = [
    ('71,72', 'Leisure & Hospitality', ['71', '72']),
    ('61,62', 'Education & Health', ['61', '62']),
    ('22,42-49', 'Trade/Trans/Util', ['22', '42', '44-45', '48-49']),
    ('54-56', 'Prof & Business', ['54', '55', '56']),
    ('52,53', 'Financial Activities', ['52', '53']),
]

def build_super_rows(year_cols=None):
    """Compute super-sector rows from individual NAICS sectors in df."""
    srows = []
    for naics, name, naics_list in super_groups:
        row = {'NAICS': naics, 'Industry': name}
        for col in (year_cols or []):
            row[col] = sum(df.loc[df['NAICS'] == n, col].values[0] for n in naics_list)
        if 'Feb 2026' in row and 'Feb 2025' in row:
            row['YoY Change'] = row['Feb 2026'] - row['Feb 2025']
            row['YoY %'] = (row['Feb 2026'] - row['Feb 2025']) / row['Feb 2025'] * 100
        if 'Feb 2021' in row:
            row['5yr Change'] = row['Feb 2026'] - row['Feb 2021']
            row['5yr %'] = (row['Feb 2026'] - row['Feb 2021']) / row['Feb 2021'] * 100
        srows.append(row)
    return srows

# Total Nonfarm from source data
tnf = nyc[nyc['INDUSTRY_TITLE'] == 'Total Nonfarm']
tnf26 = tnf[tnf['YEAR'] == 2026]['FEB'].values[0]
tnf25 = tnf[tnf['YEAR'] == 2025]['FEB'].values[0]
tnf21 = tnf[tnf['YEAR'] == 2021]['FEB'].values[0]

def build_tnf_row():
    return {
        'NAICS': '', 'Industry': 'TOTAL NONFARM',
        'Feb 2026': tnf26, 'Feb 2025': tnf25,
        'YoY Change': tnf26 - tnf25, 'YoY %': (tnf26 - tnf25) / tnf25 * 100,
    }

df_yoy = df.sort_values('YoY %', ascending=True).copy()
df_yoy['Industry'] = df_yoy['Industry'].map(short)

display_cols = ['NAICS', 'Industry', 'Feb 2026', 'Feb 2025', 'YoY Change', 'YoY %']
super_yoy = build_super_rows(['Feb 2026', 'Feb 2025'])
super_yoy.append(build_tnf_row())

df_display = pd.concat([df_yoy[display_cols], pd.DataFrame(super_yoy)], ignore_index=True)
display(df_display.style.format({
    'Feb 2026': '{:,}', 'Feb 2025': '{:,}', 'YoY Change': '{:+,}', 'YoY %': '{:+.2f}%'
}).hide(axis='index'))

priv26 = tnf26 - df.loc[df['NAICS'] == '92', 'Feb 2026'].values[0]
priv25 = tnf25 - df.loc[df['NAICS'] == '92', 'Feb 2025'].values[0]
print(f'Private sector: Feb 2026 = {priv26:,} | YoY change = {priv26 - priv25:+,}')

---
## Step 2.7–2.9: Five-Year Comparison Table (Q1b)

**CRITICAL CONTEXT:** Feb 2021 was still in COVID disruption. Large positive 5-year % changes reflect recovery from COVID lows, not sustainable growth rates.

In [ ]:
display_all = ['NAICS', 'Industry', 'Feb 2026', 'Feb 2025', 'YoY Change', 'YoY %',
               'Feb 2021', '5yr Change', '5yr %']
df_full = df.sort_values('YoY %', ascending=True).copy()
df_full['Industry'] = df_full['Industry'].map(short)

super_full = build_super_rows(['Feb 2026', 'Feb 2025', 'Feb 2021'])
super_full.append({
    'NAICS': '', 'Industry': 'TOTAL NONFARM',
    'Feb 2026': tnf26, 'Feb 2025': tnf25,
    'YoY Change': tnf26 - tnf25, 'YoY %': (tnf26 - tnf25) / tnf25 * 100,
    'Feb 2021': tnf21, '5yr Change': tnf26 - tnf21,
    '5yr %': (tnf26 - tnf21) / tnf21 * 100,
})

df_display = pd.concat([df_full[display_all], pd.DataFrame(super_full)], ignore_index=True)
display(df_display.style.format({
    'Feb 2026': '{:,}', 'Feb 2025': '{:,}', 'Feb 2021': '{:,}',
    'YoY Change': '{:+,}', '5yr Change': '{:+,}',
    'YoY %': '{:+.2f}%', '5yr %': '{:+.2f}%'
}).hide(axis='index'))

---
## Step 2.10: Chart 1 — YoY % Change by Industry

All 18 individual NAICS sectors plus 5 BLS supersectors and Total Nonfarm.

In [ ]:
chart_rows = []
for _, row in df.sort_values('YoY %', ascending=True).iterrows():
    chart_rows.append({'label': f"{row['NAICS']}: {short[row['Industry']]}", 'YoY %': row['YoY %']})

for srow in sorted(super_yoy, key=lambda x: x['YoY %']):
    chart_rows.append({'label': f"{srow['NAICS']}: {srow['Industry']}", 'YoY %': srow['YoY %']})

chart_df = pd.DataFrame(chart_rows).sort_values('YoY %')
colors = ['#d32f2f' if x < 0 else '#2e7d32' for x in chart_df['YoY %']]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=chart_df['label'], x=chart_df['YoY %'], orientation='h',
    marker_color=colors,
    text=[f"{x:+.2f}%" for x in chart_df['YoY %']],
    textposition='outside', textfont_size=9, cliponaxis=False,
))

fig.update_layout(
    title='Year-over-Year Employment Change (Feb 2026 vs Feb 2025)<br><sup>New York City | All NAICS sectors + BLS supersectors | Data: NYS DOL CES, not seasonally adjusted</sup>',
    xaxis_title='YoY % Change', yaxis_title='', height=800, showlegend=False,
    xaxis=dict(zeroline=True, zerolinecolor='#333', zerolinewidth=1),
    margin=dict(l=220, r=60, t=80, b=50), font=dict(size=10),
)
fig.show()

---
## Step 2.11: Chart 2 — 1-Year vs 5-Year % Change Comparison

All NAICS sectors plus BLS supersectors and Total Nonfarm.

In [ ]:
# Individual sectors
comp_rows = []
for _, row in df.iterrows():
    comp_rows.append({'Sector': f"{row['NAICS']}: {short[row['Industry']]}",
                      'YoY %': row['YoY %'], '5yr %': row['5yr %']})

# Supersectors + Total Nonfarm
for srow in super_full:
    label = f"{srow['NAICS']}: {srow['Industry']}" if srow['NAICS'] else srow['Industry']
    comp_rows.append({'Sector': label, 'YoY %': srow['YoY %'], '5yr %': srow['5yr %']})

comp_df = pd.DataFrame(comp_rows).sort_values('YoY %')

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=comp_df['Sector'], x=comp_df['YoY %'], name='1-Year (Feb 2026 vs Feb 2025)',
    orientation='h', marker_color='#1565c0',
))
fig2.add_trace(go.Bar(
    y=comp_df['Sector'], x=comp_df['5yr %'], name='5-Year (Feb 2026 vs Feb 2021)',
    orientation='h', marker_color='#f57c00',
))

fig2.update_layout(
    barmode='group', height=900,
    title='1-Year vs 5-Year Employment % Change: All Sectors<br><sup>Feb 2021 was COVID-distorted — large 5-year gains reflect recovery, not organic growth</sup>',
    xaxis_title='% Change',
    margin=dict(l=220, r=60, t=80, b=50), legend=dict(x=0.55, y=0.98), font=dict(size=9),
)
fig2.show()

---
## Step 2.13: Narrative — Possible Reasons for Industry Changes

---
### YoY Changes (Feb 2026 vs Feb 2025) — Top Decliners


**Manufacturing (NAICS 31-33): -5.82% (-3,100 jobs)**

Manufacturing has been in long-term structural decline in NYC, driven by automation, offshoring of production, and the city's transition to a service-oriented economy.

**Mining/Construction (NAICS 21+23): -4.82% (-6,600 jobs)**

Construction declined likely due to higher interest rates throughout 2025 that suppressed new housing starts and commercial development, along with the completion of several major infrastructure projects.

**Leisure & Hospitality (NAICS 71+72): -2.28% (-10,000 jobs)**

*Supersector composition: Arts, Entertainment & Recreation (NAICS 71) + Accommodation & Food Services (NAICS 72). BLS Supersector 70.*

This BLS supersector shows the largest absolute decline outside of Education & Health. The post-pandemic rebound has matured and possibly overshot sustainable levels. Rising operational costs, labor shortages, and remote work reducing downtown lunch and business travel demand are now headwinds.

**Other Services (NAICS 81): -2.77% (-4,900 jobs)**

Includes repair services, personal care, and civic organizations. Inflation may be reducing consumer discretionary spending on non-essential services.

**Education & Health (NAICS 61+62): -1.62% (-21,400 jobs)**

*Supersector composition: Private Educational Services (NAICS 61) + Health Care & Social Assistance (NAICS 62). BLS Supersector 65.*

The largest absolute decline of any supersector. This is remarkable because this sector was the primary engine of job growth over the prior five years (+260,100 jobs). The reversal may reflect post-pandemic normalization as healthcare demand subsides, combined with the Kaiser Permanente nurses' strike that sidelined over 30,000 workers.

---
### YoY Changes (Feb 2026 vs Feb 2025) — Top Growers

**Information (NAICS 51): +1.93% (+4,200 jobs)**

Driven by media streaming, digital content, and AI-related services. However, the NYC Comptroller (Feb 2026) projects benchmark revisions may revise some gains downward.

**Government (NAICS 92): +1.58% (+9,500 jobs)**

Driven by local government hiring, particularly in education, as the city fills positions left vacant during pandemic-era budget constraints. This is the only sector where 1-year and 5-year growth rates are similar, indicating steady rather than recovery-driven expansion.

**Financial Activities (NAICS 52+53): +1.27% (+6,500 jobs)**

*Supersector composition: Finance & Insurance (NAICS 52) + Real Estate & Rental/Leasing (NAICS 53). BLS Supersector 55.*

Steady expansion reflecting NYC's role as a global financial center, with gains in securities, asset management, and insurance.

**Key Takeaway:** NYC Total Nonfarm fell by -39,400 jobs (-0.82%) YoY to 4,791,000. Only 3 of 10 super-sectors grew over the past year.

---
### 5-Year Changes (Feb 2026 vs Feb 2021) — Top Growers

**CRITICAL CONTEXT: Feb 2021 was still in COVID disruption.** The 5-year figures below are dominated by recovery from pandemic-era lows.

**Leisure & Hospitality (NAICS 71+72): +85.05% (+197,400 jobs)**

*Supersector composition: Arts, Entertainment & Recreation (NAICS 71) + Accommodation & Food Services (NAICS 72). BLS Supersector 70.*

The largest 5-year rebound of any sector. Feb 2021 employment was just 232,100 — devastatingly low due to COVID lockdowns. The 197,400 jobs recovered since then are almost entirely a return to pre-pandemic levels, not new growth.

**Education & Health (NAICS 61+62): +25.07% (+260,100 jobs)**

*Supersector composition: Private Educational Services (NAICS 61) + Health Care & Social Assistance (NAICS 62). BLS Supersector 65. Note: both subsectors are private sector only.*

The largest absolute 5-year gain. While partially COVID recovery, this sector also benefited from structural demand growth in healthcare, aging population needs, and increased social services funding.

**Mining/Construction (NAICS 21+23): +22.80% (+28,000 jobs)**

*Combined sector: Mining, Logging & Construction. These NAICS sectors are merged in the CES data for NYC and cannot be separated.*

Reflects recovery from a Feb 2021 trough when many construction projects were paused, plus infrastructure spending.

**Information (NAICS 51): +20.53% (+42,200 jobs)**

Recovery from COVID lows plus growth in streaming, digital media, and tech. However, the sector is now decelerating — the 1-year gain (+1.93%) is a fraction of the 5-year annualized rate.

---
### 5-Year Changes (Feb 2026 vs Feb 2021) — Top Laggards

**Finance/Insurance (NAICS 52): +4.51% (+18,300 jobs)**

Modest 5-year growth for NYC's hallmark industry, reflecting that finance was less disrupted by COVID (office-based work continued) and therefore had less recovery upside.

**Government (NAICS 92): +5.83% (+34,200 jobs)**

Relatively stable over 5 years — government employment was less affected by COVID disruptions.

**Wholesale Trade (NAICS 42): -0.51% (-700 jobs)**

The only sector to decline over both 1-year and 5-year horizons, reflecting structural shifts in distribution and e-commerce.

**The divergence is the story:** Sectors that lost the most during COVID show the largest 5-year gains (recovery) but are now declining again (maturation). Sectors that were less disrupted show modest 5-year gains and steadier current performance.


---
## Step 2.12: Chart 3 — 5-Year Employment Trends (2021–2026)

All NAICS sectors, BLS supersectors, and Total Nonfarm on a single interactive chart.
Click legend items to toggle sectors on/off for comparison.

In [ ]:
import plotly.colors as pc

years_recent = sorted([y for y in nyc['YEAR'].unique() if 2021 <= y <= 2026])

def get_feb_series(title, years):
    vals, yrs = [], []
    for yr in years:
        v = nyc[(nyc['YEAR'] == yr) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
        if len(v) > 0 and pd.notna(v.values[0]):
            vals.append(int(v.values[0]))
            yrs.append(yr)
    return yrs, vals

# Color palette
palette = pc.qualitative.Dark24 + pc.qualitative.Light24

fig3 = go.Figure()
idx = 0

# Individual NAICS sectors
for naics, title in sectors.items():
    yrs, vals = get_feb_series(title, years_recent)
    if vals:
        fig3.add_trace(go.Scatter(
            x=yrs, y=vals, mode='lines+markers',
            name=f'{naics}: {short[title]}',
            legendgroup='sectors',
            line=dict(color=palette[idx % len(palette)], width=1.5),
            marker=dict(size=5),
            visible=True,
        ))
        idx += 1

# BLS Supersectors (computed)
for naics, name, naics_list in super_groups:
    vals, yrs = [], []
    for yr in years_recent:
        total = 0
        for n in naics_list:
            v = df.loc[df['NAICS'] == n]
            if len(v) > 0:
                src_title = sectors[n]
                row_v = nyc[(nyc['YEAR'] == yr) & (nyc['INDUSTRY_TITLE'] == src_title)]['FEB']
                if len(row_v) > 0 and pd.notna(row_v.values[0]):
                    total += int(row_v.values[0])
        if total > 0:
            vals.append(total)
            yrs.append(yr)
    if vals:
        fig3.add_trace(go.Scatter(
            x=yrs, y=vals, mode='lines+markers',
            name=f'{naics}: {name}',
            legendgroup='supersectors',
            line=dict(dash='dash', width=2),
            marker=dict(size=6),
        ))
        idx += 1

# Total Nonfarm
tnf_yrs, tnf_vals = get_feb_series('Total Nonfarm', years_recent)
fig3.add_trace(go.Scatter(
    x=tnf_yrs, y=tnf_vals, mode='lines+markers',
    name='TOTAL NONFARM',
    legendgroup='total',
    line=dict(width=3, color='black'),
    marker=dict(size=7),
))

fig3.update_layout(
    title='February Employment Trends: All Sectors (2021–2026)<br><sup>Click legend items to toggle on/off | Dashed = BLS supersectors | Solid black = Total Nonfarm | Data: NYS DOL CES, NSA</sup>',
    xaxis_title='Year', yaxis_title='Employment',
    xaxis=dict(tickmode='array', tickvals=years_recent, dtick=1),
    height=700, font=dict(size=10),
    legend=dict(
        title='Click to toggle', font=dict(size=9),
        itemsizing='constant',
        groupclick='toggleitem',
    ),
    margin=dict(l=80, r=20, t=90, b=50),
    hovermode='x unified',
)
fig3.show()